<a href="https://colab.research.google.com/github/anumit2004/Attention-free-Transformer/blob/main/AFT_FULL_Stabilised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## AFT Full With Stabilised Key .

In [ ]:
import torch
import math
from torch import nn
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

### Mathematics of AFTFull

**Attention Free Transformer (AFT)** replaces the attention mechanism in transformers by computing a weighted sum of values, using weights determined by key-global bias interaction. The `AFTFull` class implements a stabilized version.

Given an input sequence $X \in \mathbb{R}^{B \times T \times D}$:
- $B$: batch size, $T$: sequence length, $D$: embedding dimension

1.  **Linear Projections:** Input $X$ is linearly projected into queries ($Q$), keys ($K$), and values ($V$) using `to_q`, `to_k`, `to_v` layers.

2.  **Positional Bias (Weight Bias):** A learnable `wbias` matrix of shape `(max_seqlen, max_seqlen)` captures relative positional importance.

3.  **Numerical Stability for Exponential Keys:** Keys $K$ are stabilized by subtracting their maximum ($K_{stable} = K - \max K_{t'}$), then exponentiated ($expK = \exp(K_{stable})$) to prevent overflow.

4.  **Weighted Sum Computation:** The core AFT computation involves a ratio:
    -   **Numerator:** Weighted sum of $expK \odot V$, where weights are derived from the exponentiated positional bias: $numerator = \exp(temp\_wbias) \text{ @ } (expK \odot V)$.
    -   **Denominator:** Normalization term, weighted sum of $expK$: $denominator = \exp(temp\_wbias) \text{ @ } expK + \epsilon$.
    -   The `weighted` term is then: $weighted = \frac{numerator}{denominator}$.

5.  **Gating Mechanism:** Queries $Q$ are passed through a sigmoid activation to form a gating mechanism: $Q_{sig} = \sigma(Q)$.

6.  **Final Output:** The gated query is element-wise multiplied with the `weighted` term, then projected back to the original dimension: $Output = Q_{sig} \odot weighted \cdot W_{project}$.

This stabilization and explicit positional weighting enable AFT to perform comparably to attention mechanisms with a different computational structure.

### The `K_stable` line is a standard deep learning computational trick for numerical stability, not part of the theoretical AFT math.

In [ ]:
# AFT Full Model Implementation :

class AFTFull(nn.Module):
    def __init__(self, max_seqlen, dim, hidden_dim=64):
        super().__init__()
        self.dim = dim
        self.hidden_dim = hidden_dim
        self.to_q = nn.Linear(dim, hidden_dim)
        self.to_k = nn.Linear(dim, hidden_dim)
        self.to_v = nn.Linear(dim, hidden_dim)
        self.project = nn.Linear(hidden_dim, dim)

        # w_{t, t'} parameter
        self.wbias = nn.Parameter(torch.Tensor(max_seqlen, max_seqlen))
        nn.init.xavier_uniform_(self.wbias)

    def forward(self, x):
        B, T, _ = x.shape
        Q = self.to_q(x)
        K = self.to_k(x)
        V = self.to_v(x)

        # Extract w_{t, t'} for the current sequence length
        temp_wbias = self.wbias[:T, :T]

        # ==========================================
        # Create a lower triangular mask. True for past/present, False for future.
        causal_mask = torch.tril(torch.ones(T, T, device=x.device)).bool()

        # Mask future positions with -infinity so exp(-inf) becomes 0
        temp_wbias = temp_wbias.masked_fill(~causal_mask, float('-inf'))
        # ==========================================

        temp_wbias = temp_wbias.unsqueeze(0)

        # \sigma_q(Q_t)
        Q_sig = torch.sigmoid(Q)

        # Subtract max for numerical stability against exp overflow
        K_stable = K - K.max(dim=1, keepdim=True)[0]
        exp_K = torch.exp(K_stable)
        exp_w = torch.exp(temp_wbias)

        # Numerator: \sum_{t'=1}^T exp(K_{t'} + w_{t,t'}) \odot V_{t'}
        # Mathematically equivalent to: exp(w) @ (exp(K) * V)
        numerator = exp_w @ (exp_K * V)

        # Denominator: \sum_{t'=1}^T exp(K_{t'} + w_{t,t'})
        denominator = exp_w @ exp_K + 1e-6

        # Y_t = \sigma_q(Q_t) \odot (Numerator / Denominator)
        Yt = Q_sig * (numerator / denominator)

        return self.project(Yt)

In [ ]:
class MLP(nn.Module):
    def __init__(self, dim, hidden_dim, dp=0.1):
        super().__init__()
        self.l1 = nn.Linear(dim, hidden_dim)
        self.g1 = nn.GELU()
        self.l2 = nn.Linear(hidden_dim, dim)
        self.d1 = nn.Dropout(dp)

    def forward(self, x):
        x = self.l1(x)
        x = self.g1(x)
        x = self.d1(x)
        return self.l2(x)

### AFT Encoder Architecture

An AFT Encoder Block combines the AFT mechanism with a feed-forward network, similar to a Transformer encoder but using AFT instead of multi-head attention. The `AFTEncoderBlock` class implements this structure:

1.  **Layer Normalization (ln1):** Input `x` is normalized.
  $$x_{norm1} = \text{LayerNorm}(x)$$

2.  **AFTFull Attention (attn):** Normalized input passes through the `AFTFull` mechanism.
  $$attn\_output = \text{AFTFull}(x_{norm1})$$

3.  **Residual Connection and Dropout (d1):** Residual connection and dropout are applied to the AFT output.
  $$x = x + \text{Dropout}(attn\_output)$$

4.  **Layer Normalization (ln2):** Result is normalized again.
  $$x_{norm2} = \text{LayerNorm}(x)$$

5.  **Multi-Layer Perceptron (mlp):** Normalized output is processed by an MLP.
  $$mlp\_output = \text{MLP}(x_{norm2})$$

6.  **Residual Connection and Dropout (d2):** Another residual connection and dropout are applied to the MLP output.
  $$out = x + \text{Dropout}(mlp\_output)$$

This sequence enables the model to capture dependencies via AFT, process representations non-linearly, and ensures stable training with residual connections and layer normalization.

In [ ]:
class AFTEncoderBlock(nn.Module):
    def __init__(self, max_seqlen, dim, hidden_dim, p=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.attn = AFTFull(max_seqlen, dim, hidden_dim)
        self.mlp = MLP(dim, hidden_dim, dp=p)
        self.d1 = nn.Dropout(p)
        self.d2 = nn.Dropout(p)

    def forward(self, x):
        x_norm = self.ln1(x)
        x = x + self.d1(self.attn(x_norm))
        x_norm = self.ln2(x)
        out = x + self.d2(self.mlp(x_norm))
        return out

### AFT Model Architecture

The `AFT` class defines the complete Attention Free Transformer model, processing an input sequence through multiple AFT Encoder Blocks and projecting the output for tasks like language modeling.

**Initialization (`__init__`):**

Uses `vocab_size`, `max_seqlen`, `dim`, `hidden_dim`, `depth`, `p` (dropout). Includes:
*   `self.embed`: `nn.Embedding` for token IDs.
*   `self.pos_embed`: Learnable `nn.Embedding` for absolute positional embeddings.
*   `self.enc`: `nn.Sequential` stack of `depth` `AFTEncoderBlock` instances.
*   `self.dec`: `nn.Linear` decoder head, projecting to `vocab_size`.

**Forward Pass (`forward`):**

Given input `x` of shape `(Batch_size, Sequence_length)`:

1.  **Positional Encoding:** Token embeddings are scaled and combined with positional embeddings.
$$x = \text{self.embed}(x) \cdot \text{math.sqrt(self.dim)} + \text{self.pos_embed(positions)}$$

2.  **Encoder Stack:** Combined embeddings pass through `self.enc` (stack of `AFTEncoderBlock`s).

3.  **Decoder Output:** Final output from the encoder stack passes through `self.dec` to produce logits for each token.

In [ ]:
class AFT(nn.Module):
    def __init__(self, vocab_size, max_seqlen, dim, hidden_dim, depth=4, p=0.1):
        super().__init__()
        self.dim = dim
        self.embed = nn.Embedding(vocab_size, dim)

        # Simple learnable absolute positional embeddings
        self.pos_embed = nn.Embedding(max_seqlen, dim)

        self.enc = nn.Sequential(*[
            AFTEncoderBlock(max_seqlen, dim, hidden_dim, p=p)
            for _ in range(depth)
        ])
        self.dec = nn.Linear(dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        device = x.device

        positions = torch.arange(0, T, device=device).unsqueeze(0).expand(B, T)
        x = self.embed(x) * math.sqrt(self.dim) + self.pos_embed(positions)

        x = self.enc(x)
        out = self.dec(x)
        return out


## Preprocessing the dataset - `WIKITEXT`

In [ ]:
class WikiTextDataset(Dataset):
    """A clean dataset class that just holds pre-processed token chunks."""
    def __init__(self, input_ids_list):
        self.input_ids = input_ids_list

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        chunk = torch.tensor(self.input_ids[idx], dtype=torch.long)
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
def prepare_dataloaders(max_seqlen=64, batch_size=16, max_samples=5000, tokenizer_name="gpt2"):
    print("Loading Wikitext : ")

    raw_dataset = load_dataset("wikitext", "wikitext-103-v1", split="train")

    print("Initializing Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Tokenizing and chunking text corpus...")
    buffer = []
    all_input_ids = []

    for item in raw_dataset:
        text = item['text']

        # Skip empty lines and Wikipedia headers
        if not text.strip() or text.startswith("="):
            continue

        tokens = tokenizer.encode(text)
        buffer.extend(tokens)

        while len(buffer) >= (max_seqlen + 1):
            all_input_ids.append(buffer[:max_seqlen + 1])
            buffer = buffer[max_seqlen:]


        if max_samples != None :
            if len(all_input_ids) >= max_samples:
                all_input_ids = all_input_ids[:max_samples]
                break

    # 1. Wrap all chunks in our dataset class
    full_dataset = WikiTextDataset(all_input_ids)

    # 2. Calculate split sizes (80% Train, 10% Validation, 10% Test)
    total_size = len(full_dataset)
    train_size = int(0.8 * total_size)
    val_size = int(0.1 * total_size)
    test_size = total_size - train_size - val_size

    # 3. Perform the random split
    generator = torch.Generator().manual_seed(42) # Seed for reproducibility
    train_data, val_data, test_data = random_split(
        full_dataset,
        [train_size, val_size, test_size],
        generator=generator
    )

    print(f"Data Split Complete: {len(train_data)} Train | {len(val_data)} Val | {len(test_data)} Test")

    # 4. Create DataLoaders
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, tokenizer

##Training the dataset  -  `WIKITEXT`


In [ ]:
# =============================================================================
# 3. TRAINING ROUTINE WITH VALIDATION
# =============================================================================

def train_on_wiki():
    # Hyperparameters
    MAX_SEQLEN = 128
    EMBED_DIM = 256
    HIDDEN_DIM = 2048
    DEPTH = 25
    BATCH_SIZE = 32
    EPOCHS = 20
    LR = 1e-4
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Fetch our newly split dataloaders
    train_loader, val_loader, test_loader, tokenizer = prepare_dataloaders(
        max_seqlen=MAX_SEQLEN,
        batch_size=BATCH_SIZE,
        max_samples=None
    )
    VOCAB_SIZE = tokenizer.vocab_size

    print(f"\nConfiguration Details:")
    print(f"-> Device: {DEVICE}")
    print(f"-> Model Vocabulary Size: {VOCAB_SIZE}")
    print("Initializing AFT Model Architecture...")

    # Build Model
    model = AFT(
        vocab_size=VOCAB_SIZE,
        max_seqlen=MAX_SEQLEN,
        dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        depth=DEPTH
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()

    print("\nBeginning Training Pipeline...")
    history = {
        'train_loss': [], 'val_loss': [],
        'train_ppl': [], 'val_ppl': [],
        'val_acc': []
    }

    for epoch in range(EPOCHS):
        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.view(-1))
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()

            if batch_idx % 20 == 0:
                print(f"Epoch {epoch+1}/{EPOCHS} | Train Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

        avg_train_loss = train_loss / len(train_loader)
        train_perplexity = math.exp(avg_train_loss)

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        correct_tokens = 0
        total_tokens = 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                outputs = model(inputs)

                loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.view(-1))
                val_loss += loss.item()

                preds = outputs.argmax(dim=-1)
                correct_tokens += (preds == targets).sum().item()
                total_tokens += targets.numel()

        avg_val_loss = val_loss / len(val_loader)
        val_perplexity = math.exp(avg_val_loss)
        val_accuracy = (correct_tokens / total_tokens) * 100


        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_ppl'].append(train_perplexity)
        history['val_ppl'].append(val_perplexity)
        history['val_acc'].append(val_accuracy)

        print(f"=== Epoch {epoch+1:02d} Complete ===")
        print(f"Train Loss: {avg_train_loss:.4f} | Train PPL: {train_perplexity:.2f} | Val Loss: {avg_val_loss:.4f} | Val PPL: {val_perplexity:.2f} | Val Accuracy: {val_accuracy:.2f}%\n")
    # Return the test_loader as well so you can run final evaluations later
    return model, tokenizer, test_loader , history

if __name__ == '__main__':
    model, tokenizer, test_loader , history = train_on_wiki()

## Ploting the loss and PPL of the training and validation

In [ ]:
def plot_training_metrics(history):
    """
    Plots Loss, Perplexity, and Accuracy from the training history.
    """
    # Set style for better looking graphs
    sns.set_theme(style="whitegrid")
    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Attention-Free Transformer (AFT) Training Performance', fontsize=16, fontweight='bold', y=1.05)

    # 1. Loss Graph
    axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='blue', marker='o')
    axes[0].plot(epochs, history['val_loss'], label='Val Loss', color='red', marker='s')
    axes[0].set_title('Cross Entropy Loss', fontsize=14)
    axes[0].set_xlabel('Epochs')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    # 2. Perplexity (PPL) Graph
    axes[1].plot(epochs, history['train_ppl'], label='Train PPL', color='blue', marker='o')
    axes[1].plot(epochs, history['val_ppl'], label='Val PPL', color='red', marker='s')
    axes[1].set_title('Perplexity (PPL)', fontsize=14)
    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel('PPL')
    axes[1].set_yscale('log') # Log scale is highly recommended for PPL as it can start very high
    axes[1].legend()

    # 3. Accuracy Graph
    axes[2].plot(epochs, history['val_acc'], label='Val Accuracy', color='green', marker='^')
    axes[2].set_title('Validation Accuracy', fontsize=14)
    axes[2].set_xlabel('Epochs')
    axes[2].set_ylabel('Accuracy (%)')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

# Run the plotting function using the history generated from train_on_wiki()
plot_training_metrics(history)

## 4. Testing the Model: Text Generation

In [ ]:

# =============================================================================
#  TEST EVALUATION
# =============================================================================

def evaluate_test_set(model, test_loader, tokenizer, device):
    print("\n--- Running Final Evaluation on Unseen Test Set ---")
    model.eval()
    criterion = nn.CrossEntropyLoss()
    VOCAB_SIZE = tokenizer.vocab_size

    test_loss = 0.0
    correct_tokens = 0
    total_tokens = 0

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(test_loader):
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass
            outputs = model(inputs)

            # Calculate loss
            loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.view(-1))
            test_loss += loss.item()

            # Calculate accuracy
            preds = outputs.argmax(dim=-1)
            correct_tokens += (preds == targets).sum().item()
            total_tokens += targets.numel()

    avg_test_loss = test_loss / len(test_loader)
    test_accuracy = (correct_tokens / total_tokens) * 100

    print(f"Final Test Loss: {avg_test_loss:.4f}")
    print(f"Final Test Accuracy: {test_accuracy:.2f}%")
    print("---------------------------------------------------\n")

# Set the device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Run the evaluation using the variables returned from train_on_wiki()
evaluate_test_set(model, test_loader, tokenizer, DEVICE)

## Text generation example :

In [ ]:
# 1. Define the generation function (with temperature for better text)
def generate_text(model, tokenizer, prompt, max_new_tokens=50, device='cpu', temperature=0.8):
    model.eval() # Set the model to evaluation mode

    # Safety check: Ensure the model knows its max_seqlen (since it wasn't in AFT __init__)
    if not hasattr(model, 'max_seqlen'):
        model.max_seqlen = 64

    encoded_prompt = tokenizer.encode(prompt, return_tensors='pt').to(device)
    generated_sequence = encoded_prompt.tolist()[0]

    print(f"\nGenerating text with prompt: '{prompt}'")

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Take only the last MAX_SEQLEN tokens if the sequence exceeds it
            current_input = torch.tensor([generated_sequence[-model.max_seqlen:]], dtype=torch.long).to(device)

            # Get predictions for the next token
            outputs = model(current_input)
            next_token_logits = outputs[0, -1, :] / temperature # Apply temperature

            # Sample the next token using multinomial distribution with temperature
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()

            generated_sequence.append(next_token)

            # Stop if EOS token is generated
            if next_token == tokenizer.eos_token_id:
                break

    return tokenizer.decode(generated_sequence)

# 2. Set the device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 3. Test the model! (Using 'model' and 'tokenizer' already in your environment)
prompt = "The quick brown fox"
generated_text = generate_text(model, tokenizer, prompt, max_new_tokens=50, device=DEVICE)
print("\nGenerated Text 1:")
print(generated_text)

prompt_2 = "Once upon a time, in a land far, far away"
generated_text_2 = generate_text(model, tokenizer, prompt_2, max_new_tokens=50, device=DEVICE)
print("\nGenerated Text 2:")
print(generated_text_2)